In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS external-data
URL "abfss://dbfscontainer@mystoragelakeadb6pmgroup.dfs.core.windows.net/external"
WITH (CREDENTIAL `adb6pm-storage-credential`)

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS external_data1
URL "abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1"
WITH (CREDENTIAL `adb6pm-storage-credential`)

In [0]:
%fs 
ls abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1

create a dataframe

In [0]:
base_dir="/mnt/files"
flight_schema='''FL_DATE DATE,OP_CARRIER STRING,OP_CARRIER_FL_NUM INT,ORIGIN STRING,
                ORIGIN_CITY_NAME STRING,DEST STRING,DEST_CITY_NAME STRING,CRS_DEP_TIME INT,DEP_TIME INT,
                WHEELS_ON INT,TAXI_IN INT,CRS_ARR_TIME INT,ARR_TIME INT,CANCELLED INT,DISTANCE INT'''

flight_time_df=(spark.read.format("json")
                .schema(flight_schema)
                .option("dateformat","m/d/y")
                .load(f"{base_dir}/jsondata/flight-time.json"))
flight_time_df.show()

In [0]:
# now saving data in delta format
flight_time_df.write.format("delta") \
                .mode("overwrite") \
                .save("abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time")

Now see the data in external1 location

In [0]:

%fs 
ls abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time

If i save a DF to a location(external)
it create 'n' no of files 
they are equal to number of partitions in DF

If you don't want those many files or partitions--->
we have coalesce() to decrease the partitions hence decreases the files

goto backend external folder-->delete all part files
now execute the following 

In [0]:
flight_time_df.coalesce(1) \
                .write.format("delta") \
                .mode("overwrite") \
                .save("abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time")

Now check again, you will see only 1 file.

In [0]:
%fs 
ls abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time

Reading delta format data from an external location
 step 1: Create external location
 step 2: Read using dataframe API
 step 3: Create External table

someone has share delta dataset at external location (or)
some other project has shared and you want to read that data

there are 2 parts here
  1)create external location and
    you should have access to external location

 2)you can read delta table from the external location



step 1: Creating external location(Already done)

step 2: Read using dataframe API


In [0]:
spark.read.format("delta") \
    .load("abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time") \
    .display() 

whenever we get data at external location for a project---> we will create external table on that data

while creating external table, we need to specify the location
as data is delta data 

-->we say "using delta

-->we are creating delta table


step 3: Creating Delta table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.demo_db.flight_time_tbl(
     FL_DATE DATE,
     OP_CARRIER STRING,
     OP_CARRIER_FL_NUM INT,
     ORIGIN STRING,
     ORIGIN_CITY_NAME STRING,
     DEST STRING,
     DEST_CITY_NAME STRING,
     CRS_DEP_TIME INT,
     DEP_TIME INT,
     WHEELS_ON INT,
     TAXI_IN INT,
     CRS_ARR_TIME INT,
     ARR_TIME INT,
     CANCELLED INT,
     DISTANCE INT
)USING DELTA
LOCATION 'abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/flight_time'


In [0]:
%sql
select * from dev.demo_db.flight_time_tbl

so here external table is creaeted

in spark, if we drop a External table -->removes only metadata(table)
                                         but not the data
                                         i.e the location remains
                                         i.e("abfss:....../external)
                                         this is the external data shared by
                                         some other team or project which
                                    we want to read but we dont want to delete

so create external table and accessing it and later
we drop the external table--->metadata will be deleted
                       but we leave the data at the external location as it is 

                       so that other teams can work on that data
